In [24]:
from embedder import Embedder
embedder = Embedder()

In [25]:
# Solution 1

query = "How does approximate nearest neighbor search work?"
query_v = embedder.encode(query)

print(query_v[0]) #-0.02058203437252893

-0.02058203437252893


In [26]:
# Load documents

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [27]:
# Solution 2

from numpy import dot

doc_name = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc_content = None

for doc in documents:
    if doc["filename"] == doc_name:
        doc_content = doc["content"]
        break

if doc_content is None:
    print(f"Document '{doc_name}' not found.")

doc_v = embedder.encode(doc_content)

dot_product = dot(query_v, doc_v)
print(dot_product) # 0.36107027225589694


0.36107027225589694


In [41]:
# Solution 3

# create chunks of documents with size 2000 and step 1000
import tqdm
import numpy as np
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

vectors = []

for chunk in tqdm.tqdm(chunks):
    chunk_v = embedder.encode(chunk["content"])
    vectors.append(chunk_v)

X = np.array(vectors)
scores = X.dot(query_v)

# Get the chunk with the highest score
idx = np.argmax(scores)
chunks[idx] # 02-vector-search/lessons/07-sqlitesearch-vector.md

100%|██████████| 295/295 [00:16<00:00, 18.12it/s]


{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [ ]:
# Solution 4

from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

vsearch_query = "What metric do we use to evaluate a search engine?"
vsearch_query_vector = embedder.encode(vsearch_query)

results = vindex.search(vsearch_query_vector, num_results=1)
results # 04-evaluation/lessons/05-search-metrics.md

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [ ]:
# Solution 5

from minsearch import Index

index = Index(text_fields=["content"])
index.fit(chunks)

text_query = "How do I store vectors in PostgreSQL?"
index_results = index.search(text_query, num_results=5)

text_query_v = embedder.encode(text_query)
vindex_results = vindex.search(text_query_v, num_results=5)

print('Text search results:', [result["filename"] for result in index_results])
print('Vector search results:', [result["filename"] for result in vindex_results])

# Result: 02-vector-search/lessons/08-pgvector.md

Text search results: ['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']
Vector search results: ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']


In [ ]:
# Solution 6

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

hybrid_search_query = "How do I give the model access to tools?"
text_results = index.search(hybrid_search_query, num_results=5) 
hybrid_search_query_vector = embedder.encode(hybrid_search_query)
vector_results = vindex.search(hybrid_search_query_vector, num_results=5)

results = rrf([vector_results, text_results])
results # 01-agentic-rag/lessons/13-function-calling.md

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 